In [1]:
# ============================================================================
# 01_collect_edgar_accounting.ipynb
# Collect firm-year accounting data from the SEC EDGAR XBRL "frames" API and
# construct the Total Accrual (TACC) and Gross-Profit-over-Assets (GPOA)
# variables that mirror the Korean thesis (Eq. 10 and Eq. 4).
# All data are free and require no API key. A descriptive User-Agent is
# mandatory and requests are throttled to <10/sec per SEC rules.
# ============================================================================

In [2]:
# --- Imports and configuration -------------------------------------------
import urllib.request, json, time, os
import pandas as pd
import numpy as np

# SEC requires a descriptive User-Agent (replace with your own contact).
HDR = {"User-Agent": "Anonymous Research anonymous@example.com"}

# XBRL coverage of smaller filers is reliable only from ~2011 onward, so we
# use fiscal years 2013-2020 to match the Korean sample window.
YEARS = list(range(2013, 2021))

DATA_DIR = "../data"
os.makedirs(DATA_DIR, exist_ok=True)

In [3]:
# --- Helper: fetch one XBRL concept for one period across all filers ------
# "frames" returns the last-filed value for every reporting entity that fits
# the requested calendrical period. Duration (flow) concepts use CY####;
# instant (stock) concepts use CY####Q4I.
def fetch_frame(concept, period):
    url = f"https://data.sec.gov/api/xbrl/frames/us-gaap/{concept}/USD/{period}.json"
    req = urllib.request.Request(url, headers=HDR)
    try:
        raw = urllib.request.urlopen(req, timeout=90).read()
        d = json.loads(raw)
    except Exception as e:
        print(f"  warn: {concept} {period} -> {e}")
        return pd.Series(dtype=float)
    # Keep the last value per CIK (frames already dedupes to one per entity).
    return pd.Series({x["cik"]: x["val"] for x in d["data"]}, dtype=float)

In [4]:
# --- Collect the raw accounting panel ------------------------------------
# Flow items over the fiscal year: net income, operating cash flow, revenue,
# cost of revenue. Stock items at year end and prior year end: total assets,
# stockholders equity. Prior-year assets are the deflator in Eq. 4 and Eq. 10.
records = []
for y in YEARS:
    ni      = fetch_frame("NetIncomeLoss", f"CY{y}");                                   time.sleep(0.3)
    ocf     = fetch_frame("NetCashProvidedByUsedInOperatingActivities", f"CY{y}");      time.sleep(0.3)
    rev     = fetch_frame("Revenues", f"CY{y}");                                        time.sleep(0.3)
    cogs    = fetch_frame("CostOfRevenue", f"CY{y}");                                   time.sleep(0.3)
    at      = fetch_frame("Assets", f"CY{y}Q4I");                                       time.sleep(0.3)
    at_prev = fetch_frame("Assets", f"CY{y-1}Q4I");                                     time.sleep(0.3)
    se      = fetch_frame("StockholdersEquity", f"CY{y}Q4I");                           time.sleep(0.3)

    df = pd.DataFrame({"ni": ni, "ocf": ocf, "rev": rev, "cogs": cogs,
                       "at": at, "at_prev": at_prev, "se": se})
    df["fyear"] = y
    df.index.name = "cik"
    records.append(df.reset_index())
    print(f"CY{y}: {len(df)} firms")

panel = pd.concat(records, ignore_index=True)
panel.to_csv(f"{DATA_DIR}/edgar_raw.csv", index=False)
print("Saved raw panel:", panel.shape)

CY2013: 9049 firms


CY2014: 8744 firms


CY2015: 8368 firms


CY2016: 7912 firms


CY2017: 7595 firms


CY2018: 7471 firms


CY2019: 7528 firms


CY2020: 7875 firms


Saved raw panel: (64542, 9)


In [5]:
# --- Map CIK to ticker ----------------------------------------------------
# The SEC ticker file lets us join the accounting panel to price data later.
req = urllib.request.Request("https://www.sec.gov/files/company_tickers.json", headers=HDR)
tk = json.loads(urllib.request.urlopen(req, timeout=30).read())
tic_df = pd.DataFrame([(int(v["cik_str"]), v["ticker"]) for v in tk.values()],
                      columns=["cik", "ticker"])
panel = panel.merge(tic_df, on="cik", how="inner")
print("Firms with a ticker:", panel["ticker"].nunique())

Firms with a ticker: 5622


In [6]:
# --- Construct accounting variables (paper Eq. 4 and Eq. 10) --------------
# Total accrual: TACC = (Net Income - Operating Cash Flow) / lagged Assets.
# This is the cash-flow-statement definition (Hribar & Collins 2002), exactly
# as adopted in the Korean thesis.
panel["tacc"] = (panel["ni"] - panel["ocf"]) / panel["at_prev"]

# Gross profit over assets: GPOA = (Revenue - COGS) / lagged Assets.
panel["gpoa"] = (panel["rev"] - panel["cogs"]) / panel["at_prev"]

# Remove divide-by-zero artifacts.
panel = panel.replace([np.inf, -np.inf], np.nan)

panel.to_csv(f"{DATA_DIR}/accounting_panel.csv", index=False)
print("Saved accounting panel:", panel.shape,
      "| firms with TACC:", panel["tacc"].notna().sum())

Saved accounting panel: (35314, 12) | firms with TACC: 25399


In [7]:
# --- Save the analysis universe (tickers with a usable TACC) --------------
uni = sorted(panel.dropna(subset=["tacc"])["ticker"].unique())
pd.Series(uni, name="ticker").to_csv(f"{DATA_DIR}/universe_tickers.csv", index=False)
print("Universe size:", len(uni))

Universe size: 4400
